# Interactive Multi-Agent Workflow - Sales Assist Tool

**Workflow:** Seller Query → Supervisory Agent → Contract Agent → Research Agent → Matching Agent → Action Agent → Results

## How to Use This Notebook

1. **Run Setup** - Install packages and initialize agents
2. **Step 1** - Ask your initial query
3. **Step 2** - Review and edit the draft email
4. **Step 3** - Confirm and send the email

---
## Setup and Environment Configuration

**Note**: The Contract Agent will automatically use cached data from `contracts_cache.json` if available!

---
## Step 0: Generate Contract Cache (First Time Only)

**IMPORTANT**: Run this cell ONCE to extract and cache all contract data.

**What it does**: Extracts text and structured fields (amount, products, dates) from contracts and saves to `contracts_cache.json`.

**Time**: ~1 minute (no LLM calls!)

**When to re-run**: Only when contract files change in `docs/` folder.

In [1]:
import os
import subprocess
import sys

# Check if cache exists
cache_exists = os.path.exists("contracts_cache.json")

if cache_exists:
    print("="*80)
    print("CONTRACT CACHE ALREADY EXISTS")
    print("="*80)
    print("Cache file found: contracts_cache.json")
    print("✓ Contracts will load instantly from cache")
    print("To regenerate cache (if contracts changed):")
    print("  1. Delete contracts_cache.json")
    print("  2. Run: python cache_contracts.py")
    print("="*80)
else:
    print("="*80)
    print("GENERATING CONTRACT CACHE")
    print("="*80)
    print("Extracting text and structured fields from contracts...")
    print("This only needs to be done ONCE.")
    
    try:
        result = subprocess.run(
            [sys.executable, "cache_contracts.py"],
            capture_output=True,
            text=True,
            timeout=300
        )
        
        if result.returncode == 0:
            print("✓ Cache generated successfully!")
            print("✓ Future runs will be 5-10x faster")
        else:
            print(f"Warning: {result.stderr}")
            print("Run manually: python cache_contracts.py")
    except Exception as e:
        print(f"Could not auto-generate: {e}")
        print("Run manually: python cache_contracts.py")
    
    print("="*80)

CONTRACT CACHE ALREADY EXISTS
Cache file found: contracts_cache.json
✓ Contracts will load instantly from cache
To regenerate cache (if contracts changed):
  1. Delete contracts_cache.json
  2. Run: python cache_contracts.py


In [2]:
# Install all requirements from requirements.txt
import sys
import subprocess

print("Installing packages from requirements.txt...")
try:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"])
    print("All packages installed successfully!\n")
except subprocess.CalledProcessError as e:
    print(f"Error installing packages: {e}\n")
except FileNotFoundError:
    print("requirements.txt file not found!\n")

# Import required libraries
import os
from dotenv import load_dotenv
import warnings
warnings.filterwarnings('ignore')

# Load environment variables
load_dotenv()

# Verify credentials
print("Environment Check:")
print(f"WATSONX_APIKEY: {'Set' if os.getenv('WATSONX_APIKEY') else 'Missing'}")
print(f"WATSONX_PROJECT_ID: {'Set' if os.getenv('WATSONX_PROJECT_ID') else 'Missing'}")
print(f"TAVILY_API_KEY: {'Set' if os.getenv('TAVILY_API_KEY') else 'Missing'}")

Installing packages from requirements.txt...
All packages installed successfully!

Environment Check:
WATSONX_APIKEY: Set
WATSONX_PROJECT_ID: Set
TAVILY_API_KEY: Set


## Initialize the Supervisory Agent

The Supervisory Agent orchestrates all other agents in the workflow.

In [8]:
from supervisory_agent import SupervisoryAgent

# Initialize the Supervisory Agent
print("Initializing Supervisory Agent...")
supervisor = SupervisoryAgent(
    apikey=os.getenv("WATSONX_APIKEY"),
    project_id=os.getenv("WATSONX_PROJECT_ID")
)
print("✓ Supervisory Agent ready\n")

Initializing Supervisory Agent...
Loaded scenario actions from docs/ScenarioActions.pdf (5143 characters)
✓ Supervisory Agent ready



---
## Interactive Workflow

### Step 1: Ask Your Initial Query

Enter your query below and run the cell to execute the full multi-agent workflow.

In [ ]:
# Enter your query here
my_query = "I'm a new seller at IBM. I recently got Confluent as a new customer and I want to understand what contracts are coming up for renewal. Are there any contracts that have already expired. Based on the CRM, Contracts and webscraped information can you then put a plan of next steps"

# You can also try these queries:
# my_query = "Can you give me an overview of all contracts and what I should do in the next 30 days?"
# my_query = "Which contracts are expiring soon and what are the next steps?"
# my_query = "I need to reach out to the CPO, can you draft me an email?"

print("="*80)
print("YOUR QUERY")
print("="*80)
print(f"\n{my_query}\n")
print("="*80)
print("Executing full workflow...")
print("="*80)

# Run the workflow
my_result = supervisor.run(
    seller_query=my_query,
    contract_file_path=None,
    partner_name="Confluent"
)

# Store result for detailed analysis in subsequent cells
print("\n" + "="*80)
print("WORKFLOW EXECUTION COMPLETE")
print("="*80 + "\n")
print("✓ All agents executed successfully")
print("✓ Results available for detailed review in cells below")
print("\nScroll down to see:")
print("  • Contract Portfolio Summary")
print("  • Partner Profile & CRM Data")
print("  • Contract-CRM Matching Results")
print("  • Action Recommendations & Reasoning")
print("  • Draft Email")
print("  • CRM Updates")

### Result Component 1: Contract Portfolio Summary

View all contracts analyzed by the Contract Agent.

In [ ]:
import json

contract_summary = my_result.get('contract_summary', {})
portfolio_summary = contract_summary.get('portfolio_summary', {})

print("="*80)
print("CONTRACT PORTFOLIO SUMMARY")
print("="*80)

if portfolio_summary:
    print(f"\nTotal Contracts: {portfolio_summary.get('total_contracts', 0)}")
    print(f"Active Contracts: {len(portfolio_summary.get('active_contracts', []))}")
    print(f"Renewal Candidates: {len(portfolio_summary.get('renewal_candidates', []))}")
    print(f"Recently Expired: {len(portfolio_summary.get('recently_expired_contracts', []))}")
    
    # Detail each contract
    all_contracts = portfolio_summary.get('contract_results', [])
    if all_contracts:
        print("\n" + "="*80)
        print("CONTRACT DETAILS")
        print("="*80)
        for i, contract in enumerate(all_contracts, 1):
            structured = contract.get('structured_summary', {})
            print(f"\n{i}. {contract.get('file_name', 'Unknown')}")
            print(f"   Product(s): {', '.join(structured.get('products', ['Unknown']))}")
            print(f"   Amount: {structured.get('amount', 'Not specified')}")
            print(f"   Start Date: {contract.get('effective_date', 'Unknown')}")
            print(f"   End Date: {contract.get('end_date', 'Unknown')}")
            print(f"   Status: {contract.get('status', 'Unknown').upper()}")
            print(f"   Days to End: {contract.get('days_to_end', 'N/A')}")
else:
    print("\nNo contract data available")

### Result Component 2: Partner Profile & CRM Data

View partner intelligence and CRM opportunities from the Research Agent.

In [ ]:
partner_profile = my_result.get('partner_profile', {})
internal_data = partner_profile.get('internal_data', {})
sales_history = internal_data.get('sales_history', {})
opportunities = sales_history.get('opportunities', [])

print("="*80)
print("PARTNER PROFILE & CRM DATA")
print("="*80)

if partner_profile:
    print(f"\nPartner: {partner_profile.get('partner_name', 'Unknown')}")
    print(f"Maturity Level: {partner_profile.get('maturity_level', 'Unknown')}")
    print(f"Sales Velocity: {partner_profile.get('sales_velocity', 'Unknown')}")
    
    if opportunities:
        print(f"\n{'='*80}")
        print("CRM OPPORTUNITIES")
        print("="*80)
        print(f"Total Opportunities: {len(opportunities)}\n")
        
        for i, opp in enumerate(opportunities, 1):
            # Format amount safely
            amount = opp.get('amount', 0)
            if isinstance(amount, (int, float)):
                amount_str = f"${amount:,.0f}"
            else:
                amount_str = str(amount) if amount else "$0"
            
            print(f"{i}. {opp.get('opportunity_name', 'Unknown')}")
            print(f"   Owner: {opp.get('owner', 'Unknown')}")
            print(f"   Stage: {opp.get('stage', 'Unknown')}")
            print(f"   Amount: {amount_str}")
            print(f"   Close Date: {opp.get('close_date', 'Unknown')}")
            print(f"   Products: {opp.get('products', 'Unknown')}")
            print(f"   Next Steps: {opp.get('next_steps', 'None specified')}")
            print()
else:
    print("\nNo partner profile data available")

### Result Component 3: Contract-CRM Matching Results

View how contracts correlate with CRM opportunities from the Matching Agent.

In [ ]:
matching_data = my_result.get('matching_data', {})

print("="*80)
print("CONTRACT-CRM MATCHING RESULTS")
print("="*80)

if matching_data and not matching_data.get('error'):
    matched = matching_data.get('matched_contracts', [])
    unmatched = matching_data.get('unmatched_contracts', [])
    
    print(f"\nMatched Contracts: {len(matched)}")
    print(f"Unmatched Contracts: {len(unmatched)}")
    
    if matched:
        print(f"\n{'='*80}")
        print("MATCHED CONTRACTS")
        print("="*80)
        for match in matched:
            contract_file = match.get('contract', {}).get('file_name', 'Unknown')
            product = match.get('contract_product', 'Unknown')
            opps = match.get('opportunities', [])
            
            print(f"\n• {contract_file} ({product})")
            for opp in opps:
                print(f"  → CRM: {opp.get('opportunity_name', 'Unknown')}")
                print(f"    Owner: {opp.get('owner', 'Unknown')}")
                print(f"    Next Steps: {opp.get('next_steps', 'None')}")
    
    if unmatched:
        print(f"\n{'='*80}")
        print("UNMATCHED CONTRACTS (No CRM Entry)")
        print("="*80)
        for unmatch in unmatched:
            contract_file = unmatch.get('contract', {}).get('file_name', 'Unknown')
            product = unmatch.get('contract_product', 'Unknown')
            print(f"• {contract_file} ({product})")
else:
    print("\nNo matching data available or error occurred")
    if matching_data.get('error'):
        print(f"Error: {matching_data['error']}")

### Result Component 4: Action Recommendations & Reasoning

View the recommended next steps and risk assessment from the Action Agent.

In [ ]:
action_recommendation = my_result.get('action_recommendation', {})

print("="*80)
print("ACTION RECOMMENDATIONS & REASONING")
print("="*80)

if action_recommendation:
    # Risk Assessment
    risk_assessment = action_recommendation.get('risk_assessment', {})
    if risk_assessment:
        print("\nRISK ASSESSMENT:")
        print(f"  Risk Level: {risk_assessment.get('risk_level', 'Unknown')}")
        print(f"  Risk Score: {risk_assessment.get('risk_score', 0)}/100")
        risk_factors = risk_assessment.get('risk_factors', [])
        if risk_factors:
            print("  Risk Factors:")
            for factor in risk_factors:
                print(f"    - {factor}")
    
    # Recommended Action
    recommended_action = action_recommendation.get('recommended_action', {})
    if recommended_action:
        print(f"\n{'='*80}")
        print("RECOMMENDED NEXT STEP")
        print("="*80)
        print(f"\n{recommended_action.get('raw_recommendation', 'No recommendation available')}")
    
    # Reasoning
    reasoning = action_recommendation.get('reasoning', '')
    if reasoning:
        print(f"\n{'='*80}")
        print("REASONING")
        print("="*80)
        print(f"\n{reasoning}")
else:
    print("\nNo action recommendations available")

### Result Component 5: Draft Email

View the draft follow-up email generated by the Action Agent.

In [ ]:
action_recommendation = my_result.get('action_recommendation', {})
draft_email = action_recommendation.get('draft_email', '')

print("="*80)
print("DRAFT FOLLOW-UP EMAIL")
print("="*80)

if draft_email:
    print(f"\n{draft_email}")
else:
    print("\nNo draft email generated")

### Result Component 6: CRM Updates

View the proposed CRM updates from the Action Agent.

In [ ]:
import json

action_recommendation = my_result.get('action_recommendation', {})
crm_updates = action_recommendation.get('crm_updates', {})

print("="*80)
print("CRM UPDATES (Demo)")
print("="*80)

if crm_updates:
    print(f"\n{json.dumps(crm_updates, indent=2)}")
else:
    print("\nNo CRM updates generated")

### Step 2: Review and Edit the Draft Email

The initial workflow generated a draft email. You can now request edits or refinements to that email.

In [6]:
# First, let's extract the draft email from the initial results
initial_email = my_result.get("action_recommendation", {}).get("draft_email", "No email generated")

print("="*80)
print("ORIGINAL DRAFT EMAIL")
print("="*80)
print(f"\n{initial_email}\n")

# Now request an edit to the email
followup_query = "Can you make the email more urgent and add a specific deadline of April 15th for the response?"

# Other follow-up examples:
# followup_query = "Can you make the email shorter and more direct?"
# followup_query = "Can you add a mention of the $500K Cognos expansion opportunity?"
# followup_query = "Can you make the tone more friendly and less formal?"
# followup_query = "Can you add a bullet list of the key contracts we need to discuss?"

print("="*80)
print("EMAIL EDIT REQUEST")
print("="*80)
print(f"\n{followup_query}\n")
print("="*80)
print("Generating edited email...")
print("="*80)

# Use LLM to edit the email based on the request
from langchain_ibm import WatsonxLLM
from langchain_core.prompts import ChatPromptTemplate
import re

llm = WatsonxLLM(
    model_id="meta-llama/llama-3-3-70b-instruct",
    url="https://us-south.ml.cloud.ibm.com",
    apikey=os.getenv("WATSONX_APIKEY"),
    project_id=os.getenv("WATSONX_PROJECT_ID"),
    params={
        "max_new_tokens": 600,
        "temperature": 0.3,
        "decoding_method": "sample",
        "stop_sequences": ["```", "\n\n\n\n"]
    }
)

edit_prompt = ChatPromptTemplate.from_template(
    "You are a professional email editor. Edit the following email based on the user's request.\n\n"
    "ORIGINAL EMAIL:\n{original_email}\n\n"
    "USER REQUEST: {edit_request}\n\n"
    "INSTRUCTIONS:\n"
    "- Generate ONLY ONE complete email (no multiple versions or alternatives)\n"
    "- Start directly with 'Subject:' - no preamble or introduction\n"
    "- Do NOT repeat phrases or content\n"
    "- End cleanly after the signature line 'IBM Seller'\n"
    "- Do NOT add explanations, meta-commentary, or markdown formatting\n"
    "- Maintain professional business tone\n\n"
    "EDITED EMAIL:"
)

formatted_prompt = edit_prompt.invoke({
    "original_email": initial_email,
    "edit_request": followup_query
})

edited_email = llm.invoke(formatted_prompt)
edited_email_text = edited_email.content if hasattr(edited_email, "content") else str(edited_email)

# Post-processing to clean up the output
def clean_email_output(email_text):
    """Clean up LLM-generated email output"""
    # Remove any leading/trailing whitespace
    email_text = email_text.strip()
    
    # Remove markdown code blocks
    email_text = re.sub(r'^```.*?\n', '', email_text, flags=re.MULTILINE)
    email_text = re.sub(r'```.*?$', '', email_text, flags=re.MULTILINE)
    
    # Remove any preamble before "Subject:"
    subject_match = re.search(r'^Subject:', email_text, flags=re.MULTILINE | re.IGNORECASE)
    if subject_match:
        email_text = email_text[subject_match.start():]
    
    # Find the first occurrence of "IBM Seller" signature and cut after it
    signature_pattern = r'(IBM Seller)'
    match = re.search(signature_pattern, email_text)
    if match:
        # Keep everything up to and including "IBM Seller"
        email_text = email_text[:match.end()]
    
    # Remove duplicate consecutive lines
    lines = email_text.split('\n')
    cleaned_lines = []
    prev_line = None
    for line in lines:
        if line.strip() != prev_line:
            cleaned_lines.append(line)
            prev_line = line.strip()
    
    email_text = '\n'.join(cleaned_lines)
    
    # Final cleanup: remove excessive blank lines (more than 2 consecutive)
    email_text = re.sub(r'\n{3,}', '\n\n', email_text)
    
    return email_text.strip()

# Clean the output
edited_email_text = clean_email_output(edited_email_text)

print("\n" + "="*80)
print("EDITED EMAIL")
print("="*80 + "\n")
print(edited_email_text)

ORIGINAL DRAFT EMAIL

Subject: Urgent: Confluent_IBM-1.30.2025 Contract Renewal

Dear Chief Procurement,

I am writing to bring to your attention the urgent need to renew our contract, Confluent_IBM-1.30.2025, which expired 67 days ago. With a value of $500,092.80, this renewal opportunity is at risk if not addressed promptly. Our discussions regarding the renewal are ongoing, and I would like to request an update on the current status.

As we move forward, it is crucial that we finalize the renewal to prevent any potential competitor entry during this gap period. I have been informed that the CFO has signed 1-year renewals, and I would appreciate any insight you can provide on the next steps.

I would appreciate the opportunity to discuss this further with you and explore ways to expedite the renewal process. I will be contacting Anand Das, the opportunity owner, to get a status update today.

Best regards,
John Doe
IBM Seller
```


Here is the rewritten response:


Subject:

EMAIL ED